In [1]:
from catmap import ReactionModel

mkm_file = 'zxycracking.mkm'
model = ReactionModel(setup_file=mkm_file)
model.output_variables += ['production_rate','rate','rate_control','coverage','selectivity_control','rxn_order']
model.run()

from catmap import analyze
vm = analyze.VectorMap(model)
vm.plot_variable = 'rate' #tell the model which output to plot
vm.log_scale = True #rates should be plotted on a log-scale
vm.min = 1e-25 #minimum rate to plot
vm.max = 1e3 #maximum rate to plot
vm.plot(save='rate.pdf') #draw the plot and save it as "rate.pdf"

vm.unique_only = False
vm.plot(save='all_rates.pdf')
vm.unique_only = True

vm.production_rate_map = model.production_rate_map #attach map
vm.threshold = 1e-30 #do not plot rates below this
vm.plot_variable = 'production_rate'
vm.plot(save='production_rate.pdf')

vm.descriptor_labels = ['H adsorption [eV]', 'CH3CH2CH adsorption [eV]']
vm.subplots_adjust_kwargs = {'left':0.2,'right':0.8,'bottom':0.15}
vm.plot(save='pretty_production_rate.pdf')

vm.plot_variable = 'coverage'
vm.log_scale = False
vm.min = 0
vm.max = 1
vm.plot(save='coverage.pdf')

vm.include_labels = ['CH3CH2CH_s']
vm.plot(save='CH3CH2CH_coverage.pdf')

sa = analyze.ScalingAnalysis(model)
sa.plot(save='scaling.pdf')



If you find CatMAP useful to your research, please cite both the following papers:

 Medford, A. J., Shi, C., Hoffmann, M. J., Lausche, A. C., Fitzgibbon, S. R., Bligaard, T., & Nørskov, J. K. (2015). CatMAP: a software package for descriptor-based microkinetic mapping of catalytic trends. Catalysis Letters, 145, 794-807. 
 Vijay, S., H. Heenen, H., Singh, A. R., Chan, K., & Voss, J. (2024). Number of sites-based solver for determining coverages from steady-state mean-field micro-kinetic models. Journal of Computational Chemistry, 45(9), 546-551.

header_evaluation: fail - could not save coverage_map = [[[4.0, -3.0], [mpf('0.99999984421302378609801926381099847396793722981319110419094748840466683098422466808851504685023948944410345016189447043689865793'), mpf('0.000000000000000026428492682652041977470009913757513058591040971184278578840883530682764248572142843637125314862404838431952307000499026498914059'), mpf('0.0000000000000003313288193699783174368878072157142200705004833394004285660

ValueError: All surfaces must have numeric descriptor values: PtZSM5

In [2]:
from ase.calculators.calculator import InputError
from glob import glob
import sys
from catmap.model import ReactionModel

model.output_variables += ['production_rate','rate','rate_control','coverage','selectivity_control','rxn_order']

output_variable = 'production_rate'
logfile = glob('*.log')
if len(logfile) > 1:
    raise InputError('Ambiguous logfile. Ensure that only one file ends with .log')
model = ReactionModel(setup_file=logfile[0])

if output_variable == 'rate_control':
    dim = 2
else:
    dim = 1

labels = model.output_labels['production_rate']

def flatten_2d(output):
    "Helper function for flattening rate_control output"
    flat = []
    for x in output:
        flat+= x
    return flat

#flatten rate_control labels
if output_variable == 'rate_control':
    flat_labels = []
    for i in labels[0]:
        for j in labels[1]:
            flat_labels.append('d'+i+'/d'+j)
    labels = flat_labels

#flatten elementary-step specific labels
if output_variable in ['rate','rate_constant','forward_rate_constant','reverse_rate_constant']:
    str_labels = []
    for label in labels:
        states = ['+'.join(s) for s in label]
        if len(states) == 2:
            new_label = '<->'.join(states)
        else:
            new_label = states[0]+'<->'+states[1]+'->'+states[2]
        str_labels.append(new_label)
    labels = str_labels

table = '\t'.join(list(['descriptor-'+d for d in model.descriptor_names])+list(labels))+'\n'

for pt, output in getattr(model,output_variable+'_map'):
    if dim == 2:
        output = flatten_2d(output)
    table += '\t'.join([str(float(i)) for i in pt+output])+'\n'

f = open(output_variable+'_table.txt','w')
f.write(table)
f.close()

In [7]:
from catmap.model import ReactionModel

model = ReactionModel(setup_file='zxycracking.log')

#for MgO, cvgs in model.coverage_map:
   # print( 'descriptors:', MgO)
   #print( 'coverages', cvgs)
    
labels = model.output_labels['coverage']
for MgO ,cvg in model.coverage_map:
    print( 'descriptors',MgO)
    print( 'intermediates',labels)
    print( 'coverages', [float(c) for c in cvg])

descriptors [4.0, -3.0]
intermediates ('C3H6_s', 'C3H8_s', 'C6H13_s', 'C6H14_s', 'CH3CH2CH_s', 'H_s')
coverages [0.9999998442130238, 2.6428492682652043e-17, 3.3132881936997833e-16, 1.504289744339399e-11, 1.5577175338243003e-07, 1.4951153198578176e-31, 1.795762712369321e-13]
descriptors [4.0, -2.6363636363636367]
intermediates ('C3H6_s', 'C3H8_s', 'C6H13_s', 'C6H14_s', 'CH3CH2CH_s', 'H_s')
coverages [0.9999999397922416, 3.357422177016479e-16, 1.5112430119288858e-13, 2.5288128757916714e-10, 5.994604065608806e-08, 7.129409004635773e-32, 8.685054879269535e-12]
descriptors [4.0, -2.272727272727273]
intermediates ('C3H6_s', 'C3H8_s', 'C6H13_s', 'C6H14_s', 'CH3CH2CH_s', 'H_s')
coverages [0.9999999722589765, 4.265200856763013e-15, 6.795132129872237e-13, 4.251105334750752e-09, 2.3069189041054566e-08, 3.448607804635508e-30, 4.200453265666455e-10]
descriptors [4.0, -1.9090909090909092]
intermediates ('C3H6_s', 'C3H8_s', 'C6H13_s', 'C6H14_s', 'CH3CH2CH_s', 'H_s')
coverages [0.9999998993400344, 5.4